# Source-Term Identification — Joint (D, κ, f) Inference with KLE

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmhobbs96/pift-od-il-inverse-problems/blob/main/examples/09_inverse_source.ipynb)

This notebook reproduces **Example 3b, Figure 4** from Alberts & Bilionis (2023).

We extend the inverse parameter problem (notebook 08) by also inferring the **unknown
source term** $f(x)$.  The source is parameterised via a truncated Karhunen-Loève
expansion (KLE) with a squared-exponential kernel:

$$f(x; z) = \sum_{i=1}^{n} z_i \sqrt{\lambda_i}\, \varphi_i(x)$$

where the $z_i \sim \mathcal{N}(0,1)$ are i.i.d. and $(\lambda_i, \varphi_i)$ are the
eigenpairs of the SE covariance kernel.  The full latent vector is

$$\lambda = (\log D,\ \log\kappa,\ z_1,\ldots,z_{10}) \in \mathbb{R}^{12}$$

and nested SGLD (Algorithm 3) is run with Jeffrey's priors on the log-parameters and
standard Gaussian priors on the $z_i$.

**Estimated runtime:** 40–80 minutes on CPU with the default CONFIG.

In [ ]:
%pip install -q git+https://github.com/cmhobbs96/pift-od-il-inverse-problems.git

import time
import jax
jax.config.update('jax_enable_x64', True)
import numpy as np
import matplotlib.pyplot as plt

from pipelines.phase_c_inverse_source import run_phase_c_inverse_source

print('JAX backend:', jax.default_backend(), '| devices:', jax.devices())

def show_fig(fig, dpi=120):
    import tempfile
    from IPython.display import Image, display
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        fig.savefig(f.name, dpi=dpi, bbox_inches='tight')
        plt.close(fig)
        display(Image(f.name))

## Configuration

In [ ]:
CONFIG = {
    'D_true':               0.1,
    'kappa_true':           1.0,
    'beta':                 1e3,
    'n_obs':                40,
    'noise_std':            0.01,
    'K':                    20,
    'kle_lengthscale':      0.3,        # SE kernel lengthscale                  [0.05, 1.0]
    'kle_n_terms':          10,         # KLE truncation                         [2, 30]
    'kle_n_grid':           200,        # Nystrom grid                           [50, 1000]
    'warmup_steps':         20000,
    'outer_steps':          5000,
    'outer_step_size0':     1e-5,
    'inner_step_size0':     1e-7,
    'inner_T_prior':        10,
    'inner_T_posterior':    1,
    'n_quad':               1,
    'n_grid':               200,
    'burn_in_frac':         0.3,
    'max_condition_number': 100.0,
    'lambda0_log_D':        0.0,
    'lambda0_log_kappa':    0.5,
}

## Run Nested SGLD

In [ ]:
t_start = time.perf_counter()

src_result = run_phase_c_inverse_source(
    cfg=CONFIG,
    device_preference=jax.default_backend(),
    save_outputs=False,
)

elapsed = time.perf_counter() - t_start
print(f'Status:  {src_result["status"]}')
print(f'Runtime: {elapsed:.1f} s  ({elapsed/60:.1f} min)')

## Results

In [ ]:
# Posterior summary for D and kappa
D_post     = src_result.get('D_posterior', {})
kappa_post = src_result.get('kappa_posterior', {})
f_mean     = src_result.get('f_mean')
f_truth    = src_result.get('f_truth')
x_grid     = src_result.get('x_grid')

print('Ground truth:')
print(f'  D_true     = {CONFIG["D_true"]}')
print(f'  kappa_true = {CONFIG["kappa_true"]}')
print()
print('Posterior summary:')
if D_post:
    print(f'  D:     mean={D_post.get("mean", float("nan")):.4f}  '
          f'std={D_post.get("std", float("nan")):.4f}')
if kappa_post:
    print(f'  kappa: mean={kappa_post.get("mean", float("nan")):.4f}  '
          f'std={kappa_post.get("std", float("nan")):.4f}')

if f_mean is not None and f_truth is not None:
    source_l2 = float(np.sqrt(np.mean((np.asarray(f_mean) - np.asarray(f_truth))**2)))
    print(f'  Source L2 error (mean f vs truth): {source_l2:.4f}')

# Plot source reconstruction
if f_mean is not None and x_grid is not None:
    f_std_arr   = src_result.get('f_std')
    f_mean_arr  = np.asarray(f_mean)
    x_grid_arr  = np.asarray(x_grid)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Source reconstruction
    if f_std_arr is not None:
        f_std_np = np.asarray(f_std_arr)
        axes[0].fill_between(x_grid_arr,
                             f_mean_arr - 1.645 * f_std_np,
                             f_mean_arr + 1.645 * f_std_np,
                             alpha=0.3, color='steelblue', label='90% CI')
    axes[0].plot(x_grid_arr, f_mean_arr, 'b-', lw=2, label='Posterior mean f')
    if f_truth is not None:
        axes[0].plot(x_grid_arr, np.asarray(f_truth), 'k--', lw=1.8, label='Truth f(x)=cos(4x)')
    axes[0].set_xlabel('x')
    axes[0].set_ylabel('f(x)')
    axes[0].set_title('Source Reconstruction')
    axes[0].legend(fontsize=9)

    # Field reconstruction
    phi_truth_arr = src_result.get('phi_truth')
    phi_post_arr  = src_result.get('phi_posterior_samples')
    if phi_truth_arr is not None:
        axes[1].plot(x_grid_arr, np.asarray(phi_truth_arr), 'k--', lw=1.8, label='Truth phi')
    if phi_post_arr is not None and len(phi_post_arr) > 0:
        phi_post_np = np.asarray(phi_post_arr)
        phi_pm = phi_post_np.mean(axis=0)
        phi_ps = phi_post_np.std(axis=0)
        axes[1].fill_between(x_grid_arr, phi_pm - 1.645*phi_ps, phi_pm + 1.645*phi_ps,
                             alpha=0.3, color='darkorange', label='90% CI')
        axes[1].plot(x_grid_arr, phi_pm, color='darkorange', lw=2, label='Posterior mean phi')
    axes[1].set_xlabel('x')
    axes[1].set_ylabel('phi(x)')
    axes[1].set_title('Field Reconstruction')
    axes[1].legend(fontsize=9)

    fig.suptitle('Source-Term Identification: f(x) and \u03c6(x) Posteriors', fontsize=12)
    fig.tight_layout()
    show_fig(fig)
else:
    print('Source reconstruction arrays not available.')

# Lambda trace
lambda_chain = src_result.get('lambda_chain')
if lambda_chain is not None and len(lambda_chain) > 0:
    lc = np.asarray(lambda_chain)
    fig2, axes2 = plt.subplots(1, 2, figsize=(11, 3.5))
    axes2[0].plot(lc[:, 0], lw=0.7, color='steelblue', alpha=0.9)
    axes2[0].axhline(np.log(CONFIG['D_true']), color='tomato', lw=1.5,
                     linestyle='--', label='log(D_true)')
    axes2[0].set_xlabel('Outer step')
    axes2[0].set_ylabel('log(D)')
    axes2[0].set_title('\u03bb trace: log(D)')
    axes2[0].legend(fontsize=9)

    axes2[1].plot(lc[:, 1], lw=0.7, color='darkorange', alpha=0.9)
    axes2[1].axhline(np.log(CONFIG['kappa_true']), color='tomato', lw=1.5,
                     linestyle='--', label='log(kappa_true)')
    axes2[1].set_xlabel('Outer step')
    axes2[1].set_ylabel('log(\u03ba)')
    axes2[1].set_title('\u03bb trace: log(\u03ba)')
    axes2[1].legend(fontsize=9)

    fig2.suptitle('Outer Chain \u03bb Traces (D and \u03ba)', fontsize=11)
    fig2.tight_layout()
    show_fig(fig2)

## Interpretation

This problem has a **12-dimensional outer latent** $(\log D, \log\kappa, z_1, \ldots, z_{10})$,
making it substantially harder to explore than Example 3a (2-dimensional).

**Source L2 error** measures how well the posterior mean $\bar{f}(x)$ recovers the ground
truth $f(x) = \cos(4x)$.  Values below 0.1 indicate a good reconstruction; the quality
depends on the KLE energy fraction (how much variance the 10 kept terms capture) and on
how many outer steps the chain has taken.

**Convergence tips:** if the $\lambda$ traces are still drifting, increase `outer_steps`
(e.g., 10 000–20 000) or raise `inner_T_posterior` to 5.  The source reconstruction
improves noticeably once the $(\log D, \log\kappa)$ components have converged, because an
accurate physics model constrains the $z_i$ more effectively.